# Processing the data

Here is how you would train a sequence classifier on one batch:

In [1]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
batch["labels"] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training the model on two sentences is not going to yield very good results. To get better results, you will need to prepare a bigger dataset.

In this section we will use an example MRPC(Microsoft Research Paraphrase Corpus) dataset.
- The data set consists of 5,801 pairs of sentences, with a label indicating if they are paraphrases or not (i.e., both sentences mean the same thing).
- This is a small dataset, so it is easy to experiment with training on it.
- One of the 10 datasets composing the GLUE benchmark, which is an academic benchmark that is used to measure the performance of ML models across 10 different text vlassification tasks.

## Loading a dataset from the Hub

The Hub doesn't just contain models; it also has multiple datasets in lots of different languages.

The Datasets library provides a very simple command to download and cache a dataset on the Hub.

Here is how we can download the MRPC dataset:

In [2]:
from datasets import load_dataset

raw_datasets = load_dataset('glue','mrpc')
raw_datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

We get a `DatasetDict` object which contains:
- the training set
- the validation set
- and the test set

Each of these contains several columns (`sentence1`, `sentence2`, `label`, and `idx`) and a variable number of rows, which are *the number of elements in each set*.
- So there are 3,668 pairs of sentences in the training set,
- 408 in the validation set
- 1,725 in the test set

We can access each pair of sentences in our `raw_datasets` object by indexing, like you would a dictionary:

In [3]:
raw_train_dataset = raw_datasets['train']
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

We can see the labels are already integers, so we won't have to do any preprocessing there.

To know which integer corresponds to which label, we can inspect the `features` of our `raw_train_dataset`. This will tell us the type of each column:

In [4]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

Behind the scenes, `label` is of type `ClassLabel`, and the mapping of integers to label name is stored in the *names* folder. 
- `0` corresponds to `not_equivalent`
- `1` corresponds to 'equivalent`

### Ex.) Examining a training dataset

Let's look at element 15 of the training set and element 87 of the validation set.

In [5]:
# Looking at training dataset element 15
raw_train_dataset[15]

{'sentence1': 'Rudder was most recently senior vice president for the Developer & Platform Evangelism Business .',
 'sentence2': 'Senior Vice President Eric Rudder , formerly head of the Developer and Platform Evangelism unit , will lead the new entity .',
 'label': 0,
 'idx': 16}

In [6]:
# Looking at validation dataset element 87
raw_valid_dataset = raw_datasets['validation']
raw_valid_dataset[87]

{'sentence1': 'However , EPA officials would not confirm the 20 percent figure .',
 'sentence2': 'Only in the past few weeks have officials settled on the 20 percent figure .',
 'label': 0,
 'idx': 812}

In [7]:
raw_valid_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

## Preprocessing a dataset

To preprocess the dataset, we need to convert the text to numbers the model can make sense of. This is done with a tokenizer.
- We can feed the tokenizer one sentence or a list of sentences, so we can directly tokenize all the first sentences and all the second sentences of each pair like this:

In [8]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_train_dataset_s1 = list(raw_train_dataset["sentence1"])
raw_train_dataset_s2 = list(raw_train_dataset["sentence2"])

tokenized_sentences_1 = tokenizer(raw_train_dataset_s1)
tokenized_sentences_2 = tokenizer(raw_train_dataset_s2)


We can't just pass two sequences to the model and get a prediction of whether the two sentences are paraphrases or not. We need to handle the two sequences as a pair, and apply the appropriate preprocessing.

Fortunately, the tokenizer can also take a pair of sequences and prepare it in the way our BERT model expects:

In [18]:
inputs = tokenizer(raw_train_dataset_s1,raw_train_dataset_s2)

- **input_ids**: Ids of the tokens from the tokenized input strings.
- **token_type_ids**: In this example, this is what tells the model which part of the input is in the first sentence and which is in the second sentence.
- **attention_mask**: Indicates wether to pay 'attention' to the token at that index
  

If we decode the IDs inside `input_ids` back to words:

In [10]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]) #Only converts the first element pair (s1,s2) which is found at index 0

['[CLS]',
 'am',
 '##ro',
 '##zi',
 'accused',
 'his',
 'brother',
 ',',
 'whom',
 'he',
 'called',
 '"',
 'the',
 'witness',
 '"',
 ',',
 'of',
 'deliberately',
 'di',
 '##stor',
 '##ting',
 'his',
 'evidence',
 '.',
 '[SEP]',
 'referring',
 'to',
 'him',
 'as',
 'only',
 '"',
 'the',
 'witness',
 '"',
 ',',
 'am',
 '##ro',
 '##zi',
 'accused',
 'his',
 'brother',
 'of',
 'deliberately',
 'di',
 '##stor',
 '##ting',
 'his',
 'evidence',
 '.',
 '[SEP]']

We see the model expects the inputs to be of the form `[CLS] sentence1 [SEP] sentence2 [SEP]` when there are two sentences.

Aligning this with the `token_type_ids` gives us:

In [11]:
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])) #Only converts the first element pair (s1,s2) which is found at index 0
print(inputs['token_type_ids'][0])

['[CLS]', 'am', '##ro', '##zi', 'accused', 'his', 'brother', ',', 'whom', 'he', 'called', '"', 'the', 'witness', '"', ',', 'of', 'deliberately', 'di', '##stor', '##ting', 'his', 'evidence', '.', '[SEP]', 'referring', 'to', 'him', 'as', 'only', '"', 'the', 'witness', '"', ',', 'am', '##ro', '##zi', 'accused', 'his', 'brother', 'of', 'deliberately', 'di', '##stor', '##ting', 'his', 'evidence', '.', '[SEP]']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


As we can see, the parts of the input corresponding to `[CLS] sentence1 [SEP]` all have a token type ID of `0`, while the other parts corresponding to `sentence2 [SEP]`, all have a token type ID of `1`.

Note: If you select a different checkpoint, you won't necessarily have the `token_type_ids` in your tokenized inputs (for instance, they're not returned if you use a DistilBERT model).

They are only returned *when the model will know what to do with them, because it has seen them during its pretraining.*

Here, BERT is pretrained with token type IDs, and on top of the masked language modeling objective, it has an additional objective called **next sentence prediction**. The goal with this task is to model to relationship between pairs of sentences.

With next sentence prediciton, the model is provided pairs of sentences (with randomly masked tokens) and asked to predict whether the second sentence follows the first. To make the task non-trivial, half of the time the sentences follow each other in the original document they were extracted from, and the other half of the time the two sentences come from two different documents.

In general, you don't need to worry about whether or not there are `token_type_ids` in your tokenized inputs: as long as you use the same checkpoint for the tokenizer and the model, everything will be fine as the tokenizer knows what to provide to its model.

Now that we have seen how our tokenizer can deal with one pair of sentences, we can use it to tokenize our who dataset:

We can feed the tokenizer a list of pairs of sentences by giving it the list of first sentences, then the list of second sentences. This is also compatibale with the padding and truncation options of the tokenizer.

In [12]:
tokenized_dataset = tokenizer(
    raw_train_dataset_s1,
    raw_train_dataset_s2,
    padding=True,
    truncation=True,
)

This works well, but it has the disadvantage of returning a dictionary (with our keys, `input_ids`, `attention_mask`, and `token_type_ids`, and values that are lists of lists).
It will also only work if you have enough RAM to store your whole dataset during the tokenization (whereas the datasets from the *Datasets* library are Apache Arrow files stored on the disk, so you only keep the same you ask for loaded in memory).

To keep the data as a dataset, we will use the `Dataset.map()` method.
This allows us some extra flexibility, if we need more preprocessing done than just tokenization. The `map()` method works by applying a function on each element of the dataset, so let's define a function that tokenizes our inputs

In [13]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

This function takes a dictionary (like the items of our dataset) and returns a new dictionary with the keys `input_ds`, `attention_mask`, and `token_type_ids`. 

Note that it also works if the `example` dictionary contains several samples (each key as a list of sentences) since the `tokenizer` works on lists of pairs of sentences, as seen before. This will allow us to use the option `batched=True` in our call to `map()`, which will greatly speed up the tokenization.

The `tokenizer` is backed by a tokenizer written in Rust from the `Tokenizers` library. This tokenizer can be very fast, but only if we give it lots of inputs at once.

Note that we have left the `padding` argument out in our tokenization function for now. This is because padding all the samples to the maximum length is not efficient: it's better to pad the samples when we're building a batch, as then we only need to *pad to the maximum length in that batch, and not the maximum length in the entire dataset*. This can save a lot of time and processing power when the inputs have  very variable lengths!

Here is how we apply the tokenization function on all our datasets at once. We're using `batched=True` in our call to `map` so the function is applied to multiple elements of our dataset at once, and not on each element separately. This allows for faster processing.

In [14]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

You can even use multiprocessing when applying your preprocessing function with `map()` by passing along a `num_proc` argument. We didn't do this here beacuase the Tokenizers library already uses multiple threads to tokenize our samples faster, but if you are not using a fast tokenizer backed by this library, this could speed up your preprocessing.

Our `tokenize_function` returns a dictionary with the keys `input_ids`, `attention_mask`, and `token_type_ids`, so those three fields are added to all splits of our dataset. Note that we could also have changed existing fields if our preprocessing function returned a new value for an existing key in the dataset to which we applied `map()`.

The last thing we will need to do is pad all the examples to the length of the longest element when we batch elements together - a technique that is referred to as *dynamic padding*.

### Dynamic Padding

The function that is responsible for putting together samples inside a batch is called a *collate function*. It's an argument you can pass when you build a `DataLoader`, the default being a function that will just convert your samples to PyTorch tensors and concatenate them (recursively if your elements are lists, tuples, or dictionaries).

This won't be possible in our case because the inputs we have won't all be of the same size. We have deliberately postponed the padding, to only apply it as necessary on each batch and avoid having over-long inputs with a lot of padding. This will speed up training by quite a bit, but note that if you're training on a TPU it can cause problems - TPUs prefer fixed shapes, even when that requires extra padding.

To do this in practice, we have to define a collate function that will apply the correct amount of padding to the items of the dataset we want to batch together. Fortunately, the Transformers library provides us with such a function via `DataCollatorWithPadding`. It takes a tokenizer when you instantiate it (in order to know which padding token to use, and whether the model expects padding to be on the left or on the right of the inputs) and will do everything you need:

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

To test this new toy, let's grab a few samples from our training set that we would like to batch together. Here we remove the columns `idx`, `sentence1`, and `sentence2` as they won't be needed and contain strings (and we can't create tensors with strings) and have a look at the lengths of each entry in the batch:

In [16]:
samples = tokenized_datasets["train"][:8]

samples = {k: v for k,v in samples.items() if k not in ["idx", "sentence1", "sentence2"]} # Tokenized data set already has the strings tokenized.

[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

We get samples of varying length, from 32 to 67. Dynamic padding means that the samples in this batch should all be padded to a length of 67, the maximum legnth inside the batch. 

Without dynamic padding, all of the samples would have to be padded to the maximum legnth in the whole dataset, or the maximum length the model can accept. Let's double-check that our `data_collator` is dynamically padding the batch properly: 

In [17]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

Now that we've gone from raw text to batches our model can deal with, we're ready to fine-tune it!

## Key Takeaways:

- Use `batched=True` with `Dataset.map()` for significantly faster preprocessing
- Dynamic padding with `DataCollatorWithPadding` is more efficient than fixed-length padding
- Always preprocess your data to match what your model expects (numerical tensors, correct column names)
- The Datasets library provides powerful tools for efficient data processing at scale.